In [15]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=88888888
    database="fraud_analysis"
)

print("MySQL Connected Successfully!")

MySQL Connected Successfully!


In [17]:
import pandas as pd

df = pd.read_csv('credit_card_fraud_cleaned.csv')

print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'credit_card_fraud_cleaned.csv'

In [18]:
import os

print(os.getcwd())

C:\Users\ramye\Downloads\DA Project\SQL


In [19]:
os.listdir()

['.ipynb_checkpoints', 'SQL-python connectivity.ipynb']

In [20]:
import os

for root, dirs, files in os.walk(os.path.expanduser("~")):
    if 'credit_card_fraud_cleaned.csv' in files:
        print(os.path.join(root, 'credit_card_fraud_cleaned.csv'))

C:\Users\ramye\Downloads\DA Project\credit_card_fraud_cleaned.csv


In [21]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ramye\Downloads\DA Project\credit_card_fraud_cleaned.csv")

print(df.shape)

(500000, 20)


In [22]:
print(df.columns.tolist())

['transaction_id', 'customer_id', 'transaction_datetime', 'transaction_date', 'transaction_hour', 'merchant', 'merchant_category', 'transaction_amount', 'payment_method', 'transaction_status', 'city', 'state', 'customer_latitude', 'customer_longitude', 'merchant_latitude', 'merchant_longitude', 'city_population', 'is_fraud', 'day_of_week', 'amount_range']


In [23]:
cursor = conn.cursor()

create_table_query = """
CREATE TABLE IF NOT EXISTS credit_card_transactions (
    transaction_id VARCHAR(50),
    customer_id VARCHAR(50),
    transaction_datetime DATETIME,
    transaction_date DATE,
    transaction_hour INT,
    merchant VARCHAR(100),
    merchant_category VARCHAR(100),
    transaction_amount DECIMAL(12,2),
    payment_method VARCHAR(50),
    transaction_status VARCHAR(50),
    city VARCHAR(100),
    state VARCHAR(100),
    customer_latitude DOUBLE,
    customer_longitude DOUBLE,
    merchant_latitude DOUBLE,
    merchant_longitude DOUBLE,
    city_population BIGINT,
    is_fraud INT,
    day_of_week VARCHAR(20),
    amount_range VARCHAR(50)
)
"""

cursor.execute(create_table_query)

print("Table created successfully!")

Table created successfully!


In [24]:
import pandas as pd
import numpy as np

# Convert NaN values to None for MySQL
df = df.replace({np.nan: None})

insert_query = """
INSERT INTO credit_card_transactions VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
"""

batch_size = 5000

for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start + batch_size]

    cursor.executemany(insert_query, batch.values.tolist())
    conn.commit()

    print(f"Inserted {start + len(batch)} / {len(df)} rows")

print("All data imported successfully!")


Inserted 5000 / 500000 rows
Inserted 10000 / 500000 rows
Inserted 15000 / 500000 rows
Inserted 20000 / 500000 rows
Inserted 25000 / 500000 rows
Inserted 30000 / 500000 rows
Inserted 35000 / 500000 rows
Inserted 40000 / 500000 rows
Inserted 45000 / 500000 rows
Inserted 50000 / 500000 rows
Inserted 55000 / 500000 rows
Inserted 60000 / 500000 rows
Inserted 65000 / 500000 rows
Inserted 70000 / 500000 rows
Inserted 75000 / 500000 rows
Inserted 80000 / 500000 rows
Inserted 85000 / 500000 rows
Inserted 90000 / 500000 rows
Inserted 95000 / 500000 rows
Inserted 100000 / 500000 rows
Inserted 105000 / 500000 rows
Inserted 110000 / 500000 rows
Inserted 115000 / 500000 rows
Inserted 120000 / 500000 rows
Inserted 125000 / 500000 rows
Inserted 130000 / 500000 rows
Inserted 135000 / 500000 rows
Inserted 140000 / 500000 rows
Inserted 145000 / 500000 rows
Inserted 150000 / 500000 rows
Inserted 155000 / 500000 rows
Inserted 160000 / 500000 rows
Inserted 165000 / 500000 rows
Inserted 170000 / 500000 rows


In [25]:
cursor.execute("""
SELECT COUNT(*) AS total_transactions
FROM credit_card_transactions
""")

result = cursor.fetchone()

print("Total Transactions:", result[0])


Total Transactions: 500000


In [26]:
cursor.execute("""
SELECT SUM(transaction_amount) AS total_transaction_amount
FROM credit_card_transactions
""")

result = cursor.fetchone()

print("Total Transaction Amount:", round(result[0], 2))

Total Transaction Amount: 35395944.09


In [27]:
cursor.execute("""
SELECT *
FROM credit_card_transactions
WHERE is_fraud = 1
""")

fraud_data_sql = cursor.fetchall()

print("Total Fraud Transactions:", len(fraud_data_sql))

Total Fraud Transactions: 2894


In [28]:
cursor.execute("""
SELECT
    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    (SUM(is_fraud) * 100.0 / COUNT(*)) AS fraud_percentage
FROM credit_card_transactions
""")

result = cursor.fetchone()

print("Total Transactions:", result[0])
print("Fraud Transactions:", result[1])
print("Fraud Percentage:", round(result[2], 2), "%")

Total Transactions: 500000
Fraud Transactions: 2894
Fraud Percentage: 0.58 %


In [29]:
cursor.execute("""
SELECT
    customer_id,
    SUM(transaction_amount) AS total_spending
FROM credit_card_transactions
GROUP BY customer_id
ORDER BY total_spending DESC
LIMIT 10
""")

result = cursor.fetchall()

for row in result:
    print(row)
    

('CUST_3F128965B93D', Decimal('121905.68'))
('CUST_A3A62304E057', Decimal('114755.28'))
('CUST_FA46922D7B8A', Decimal('113715.86'))
('CUST_8715D9E40DC2', Decimal('112027.96'))
('CUST_CD81464D3E96', Decimal('111833.19'))
('CUST_5B9E1E38487F', Decimal('111076.21'))
('CUST_82BC011F90F9', Decimal('110738.66'))
('CUST_E2097D89A4B2', Decimal('110403.78'))
('CUST_B82BF3D7EBC1', Decimal('110297.57'))
('CUST_B4FA0AABA467', Decimal('108153.52'))


In [30]:
cursor.execute("""
SELECT
    merchant_category,
    COUNT(*) AS total_transactions,
    SUM(transaction_amount) AS total_amount,
    AVG(transaction_amount) AS average_amount
FROM credit_card_transactions
GROUP BY merchant_category
ORDER BY total_amount DESC
""")

result = cursor.fetchall()

for row in result:
    print(row)

('grocery_pos', 47682, Decimal('5575257.81'), Decimal('116.925838'))
('shopping_pos', 45050, Decimal('3594166.47'), Decimal('79.781720'))
('shopping_net', 37539, Decimal('3407909.83'), Decimal('90.783181'))
('gas_transport', 50785, Decimal('3223400.84'), Decimal('63.471514'))
('home', 47281, Decimal('2772278.76'), Decimal('58.634097'))
('kids_pets', 43642, Decimal('2520498.55'), Decimal('57.753965'))
('entertainment', 36454, Decimal('2354194.17'), Decimal('64.579859'))
('misc_net', 24447, Decimal('1996390.57'), Decimal('81.661986'))
('misc_pos', 30678, Decimal('1898913.79'), Decimal('61.898226'))
('travel', 15624, Decimal('1848202.58'), Decimal('118.292536'))
('health_fitness', 33112, Decimal('1794560.25'), Decimal('54.196673'))
('food_dining', 35118, Decimal('1791253.86'), Decimal('51.006716'))
('personal_care', 34941, Decimal('1675878.67'), Decimal('47.963100'))
('grocery_net', 17647, Decimal('943037.94'), Decimal('53.438995'))


In [31]:
cursor.execute("""
SELECT
    transaction_hour,
    COUNT(*) AS total_transactions,
    SUM(transaction_amount) AS total_amount
FROM credit_card_transactions
GROUP BY transaction_hour
ORDER BY transaction_hour
""")

result = cursor.fetchall()

for row in result:
    print(row)

(0, 16379, Decimal('1311680.59'))
(1, 16560, Decimal('1317319.49'))
(2, 16407, Decimal('1319111.05'))
(3, 16627, Decimal('1323478.87'))
(4, 16143, Decimal('1230753.10'))
(5, 16316, Decimal('1240368.02'))
(6, 16361, Decimal('1243118.43'))
(7, 16376, Decimal('1221480.88'))
(8, 16454, Decimal('1269283.80'))
(9, 16142, Decimal('1210049.43'))
(10, 16259, Decimal('1235120.17'))
(11, 16212, Decimal('1238669.95'))
(12, 25004, Decimal('1536445.41'))
(13, 25171, Decimal('1591499.37'))
(14, 25025, Decimal('1597187.49'))
(15, 25458, Decimal('1604050.60'))
(16, 25381, Decimal('1650408.46'))
(17, 25142, Decimal('1637542.71'))
(18, 25640, Decimal('1622994.47'))
(19, 25049, Decimal('1591564.27'))
(20, 24847, Decimal('1615053.66'))
(21, 25151, Decimal('1606849.38'))
(22, 25952, Decimal('2119004.65'))
(23, 25944, Decimal('2062909.84'))


In [32]:
cursor.execute("""
SELECT
    customer_id,
    SUM(transaction_amount) AS total_amount,
    RANK() OVER (ORDER BY SUM(transaction_amount) DESC) AS customer_rank
FROM credit_card_transactions
GROUP BY customer_id
LIMIT 10
""")

result = cursor.fetchall()

for row in result:
    print(row)

('CUST_3F128965B93D', Decimal('121905.68'), 1)
('CUST_A3A62304E057', Decimal('114755.28'), 2)
('CUST_FA46922D7B8A', Decimal('113715.86'), 3)
('CUST_8715D9E40DC2', Decimal('112027.96'), 4)
('CUST_CD81464D3E96', Decimal('111833.19'), 5)
('CUST_5B9E1E38487F', Decimal('111076.21'), 6)
('CUST_82BC011F90F9', Decimal('110738.66'), 7)
('CUST_E2097D89A4B2', Decimal('110403.78'), 8)
('CUST_B82BF3D7EBC1', Decimal('110297.57'), 9)
('CUST_B4FA0AABA467', Decimal('108153.52'), 10)


In [33]:
cursor.execute("""
SELECT AVG(transaction_amount) AS average_transaction_amount
FROM credit_card_transactions
""")

result = cursor.fetchone()

print("Average Transaction Amount:", round(result[0], 2))

Average Transaction Amount: 70.79


In [34]:
cursor.execute("""
SELECT
    state,
    city,
    COUNT(*) AS fraud_transactions,
    SUM(transaction_amount) AS fraud_amount
FROM credit_card_transactions
WHERE is_fraud = 1
GROUP BY state, city
ORDER BY fraud_transactions DESC
LIMIT 10
""")

result = cursor.fetchall()

for row in result:
    print(row)

('TX', 'Houston', 18, Decimal('10089.92'))
('OK', 'Tulsa', 14, Decimal('9791.78'))
('TX', 'San Antonio', 13, Decimal('7048.78'))
('FL', 'Naples', 11, Decimal('6254.44'))
('MI', 'Detroit', 11, Decimal('5560.23'))
('WY', 'Freedom', 10, Decimal('4406.32'))
('OR', 'Lowell', 10, Decimal('4425.26'))
('NM', 'Albuquerque', 10, Decimal('4770.76'))
('ME', 'Nobleboro', 9, Decimal('2225.29'))
('IL', 'Bonfield', 9, Decimal('5450.83'))
